# Kestrel V3, step 0: A100 speed at seq 2k vs 4k

Measures one full training step (fwd+bwd x accum + Muon + AdamW) for Kestrel-Nano on
random tokens, at the same 196,608 tokens/step the V1 runs used. No Drive and no weights
are needed. **Cost: ~10-15 min of A100, about 1.5-2 units.**

`Runtime -> Change runtime type -> A100 GPU`, then `Run all`. Results print at the end and
are saved to `/content/step0_bench.json`.

Note: the bench runs every step at R=2 (the nano preset trains on R in {1, 2} with p=0.5),
so it is an upper bound on step time. The real-run figure (~18.5k tok/s at seq 4096)
includes R=1 steps and data loading.


In [ ]:
# 0. GPU check: refuse to burn units on anything but an A100/H100
import torch
name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU'
mem = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0
print(f'torch {torch.__version__} | {name} | {mem:.0f} GB')
if 'A100' not in name and 'H100' not in name:
    raise SystemExit(f'Got {name}, not an A100. Runtime -> Change runtime type -> A100, then rerun.')


In [ ]:
# 1. Code (public repo, Colab's preinstalled torch)
import os
REPO = '/content/Kestrel'
if not os.path.isdir(REPO):
    !git clone -q https://github.com/YoMosa2009/Kestrel.git {REPO}
else:
    !git -C {REPO} pull -q
%cd {REPO}
!git log --oneline -1


In [ ]:
# 2. The bench. Each row is one subprocess so an OOM can't poison the next one.
#    Same tokens/step everywhere: 196,608. 'ckpt1' = checkpoint every block (the V1
#    recipe), 'ckpt2' = every 2nd block, 'none' = no activation checkpointing.
import subprocess, re, json, time
ROWS = [
    # seq, batch, accum, checkpointing
    (4096,  8, 6, 'ckpt1'),
    (4096,  8, 6, 'ckpt2'),
    (4096,  8, 6, 'none'),
    (4096, 16, 3, 'ckpt1'),
    (2048, 16, 6, 'ckpt1'),
    (2048, 16, 6, 'ckpt2'),
    (2048, 16, 6, 'none'),
    (2048, 32, 3, 'ckpt1'),
]
results = []
t0 = time.time()
for seq, batch, accum, ck in ROWS:
    args = ['python', '-m', 'scripts.bench_step', '--seq', str(seq), '--batch', str(batch),
            '--accum', str(accum), '--iters', '4', '--warmup', '1']
    if ck == 'ckpt1':  args += ['--only', 'tf32+fusedadam']
    if ck == 'ckpt2':  args += ['--only', 'tf32+fusedadam+ckpt2']
    if ck == 'none':   args += ['--only', 'tf32+fusedadam', '--no-checkpoint']
    p = subprocess.run(args, capture_output=True, text=True)
    out = p.stdout + p.stderr
    m = re.search(r'([\d.]+) tok/s \| +([\d.]+) s/step \| peak +([\d.]+) GB', out)
    row = dict(seq=seq, batch=batch, accum=accum, ckpt=ck)
    if m:
        row.update(tok_s=float(m[1]), s_step=float(m[2]), peak_gb=float(m[3]))
    else:
        row.update(error='OOM' if 'OOM' in out or 'OutOfMemory' in out else out[-600:])
    results.append(row)
    print(row, f'| {time.time()-t0:.0f}s elapsed', flush=True)
json.dump(results, open('/content/step0_bench.json', 'w'), indent=1)


In [ ]:
# 3. Summary. Units per 1B tokens at ~6.8 units per A100-hour (measured in V1).
UNITS_PER_HOUR = 6.8
print(f"{'seq':>5} {'batch':>5} {'accum':>5} {'ckpt':>6} {'tok/s':>8} {'peak GB':>8} {'h/1B tok':>9} {'units/1B':>9}")
for r in results:
    if 'tok_s' in r:
        h = 1e9 / r['tok_s'] / 3600
        print(f"{r['seq']:>5} {r['batch']:>5} {r['accum']:>5} {r['ckpt']:>6} {r['tok_s']:>8.0f} "
              f"{r['peak_gb']:>8.1f} {h:>9.1f} {h*UNITS_PER_HOUR:>9.0f}")
    else:
        print(f"{r['seq']:>5} {r['batch']:>5} {r['accum']:>5} {r['ckpt']:>6}  {r['error'][:60]}")
print('STEP0_RESULTS ' + json.dumps(results))
